In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, regexp_extract, to_timestamp, date_format
from pyspark.sql.types import FloatType, DoubleType, IntegerType, TimestampType

### Data Load

In [0]:
# Bronze layer path
bronze_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/bronze/"

old_battery_data = f"{bronze_path}old_battery_data"
realtime_ingestion = f"{bronze_path}realtime_ingestion"
bms_logs = f"{bronze_path}bms_logs"
realtime_bms_logs = f"{bronze_path}realtime_bms_logs"

### Spark Dataframe Create

In [0]:
df_old_battery = spark.read.format('parquet')\
               .option('inferScehma', True)\
               .load(old_battery_data)

df_realtime_battery = spark.read.format('parquet')\
               .option('inferScehma', True)\
               .load(realtime_ingestion)

df_old_bms = spark.read.format('text')\
               .option('inferScehma', True)\
               .load(bms_logs)

df_realtime_bms = spark.read.format('text')\
               .option('inferScehma', True)\
               .load(realtime_bms_logs)

####Schema Enforcement: Old Battery Data

In [0]:
df_silver_old_battery = df_old_battery.select(
            col("battery_id"),
            col("timestamp").cast(TimestampType()).alias("timestamp"),
            col("voltage").cast(DoubleType()).alias("voltage"),
            col("temperature").cast(DoubleType()).alias("temperature"),
            col("cycle_count").cast(DoubleType()).cast(IntegerType()).alias("cycle_count"),
            
)

df_silver_old_battery.toPandas().info()


#### Schema Enforcement: Old BMS Logs

In [0]:
from pyspark.sql import functions as F

df_old_bms = df_old_bms.filter(F.col("value").startswith("["))

df_old_bms = df_old_bms.withColumn(
    "logs", 
    F.regexp_extract(F.col("value"), r"\]\s*(.*)", 1) 
).withColumn(
    "timestamp", 
    F.date_format(
        F.to_timestamp(
            F.regexp_extract(F.col("value"), r"\[(.*?)\]", 1), 
            "yyyy-MM-dd HH:mm:ss"
        ), 
        "yyyy-MM-dd'T'HH:mm:ss.SSSXXX" 
    )
).drop("value")

df_old_bms.show(truncate=False)

In [0]:
df_old_bms = df_old_bms.select(
    col("logs"),
    col("timestamp").cast(TimestampType()).alias("timestamp")
).orderBy(col("timestamp"))
df_old_bms.display()

####Schema Enforcement: Realtime Battery Data

Schema Generate

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

battery_data_schema = StructType([
    StructField("battery_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("voltage", DoubleType(), True),
    StructField("temp", DoubleType(), True),
    StructField("cycle_count", IntegerType(), True)
])

In [0]:
# parse json string into json
df_realtime_battery = df_realtime_battery.withColumn("parsed_body", from_json(col("body"), battery_data_schema))


df_silver_realtime_battery = df_realtime_battery.select(
                                col("parsed_body.battery_id").alias("battery_id"),
                                col("parsed_body.timestamp").alias("timestamp"),
                                col("parsed_body.voltage").alias("voltage"),
                                col("parsed_body.temp").alias("temp"),
                                col("parsed_body.cycle_count").alias("cycle_count"))


df_silver_realtime_battery.toPandas().info()                                                    

####Schema Enforcement: Realtime BMS logs

In [0]:
from pyspark.sql import functions as F

df_realtime_bms = df_realtime_bms.filter(F.col("value").startswith("["))

df_realtime_bms = df_realtime_bms.withColumn(
    "logs", 
    F.regexp_extract(F.col("value"), r"\]\s*(.*)", 1) 
).withColumn(
    "timestamp", 
    F.date_format(
        F.to_timestamp(
            F.regexp_extract(F.col("value"), r"\[(.*?)\]", 1), 
            "yyyy-MM-dd HH:mm:ss"
        ), 
        "yyyy-MM-dd'T'HH:mm:ss.SSSXXX" 
    )
).drop("value")

df_realtime_bms.show(truncate=False)

In [0]:
df_realtime_bms = df_realtime_bms.select(
    col("logs"),
    col("timestamp").cast(TimestampType()).alias("timestamp")
).orderBy(col("timestamp"))
df_realtime_bms.display()

Special case

In [0]:
df_old_bms_grouped = df_old_bms.withColumn("join_hour", F.date_trunc("hour", col("timestamp"))).groupBy("join_hour").agg(F.collect_list("logs").alias("logs"))


df_old_battery_grouped = df_silver_old_battery.withColumn("join_hour", F.date_trunc("hour", col("timestamp")))

In [0]:
df_silver_old_final = df_old_battery_grouped.join(df_old_bms_grouped, on="join_hour", how="full")
df_silver_old_final = df_silver_old_final.withColumn(
    "timestamp", 
    F.coalesce(
        F.col("timestamp"), 
        F.col("join_hour")
    )
).drop("join_hour")
df_silver_old_final.toPandas().info()

In [0]:
df_silver_old_final.display()

In [0]:
df_realtime_bms_grouped = df_realtime_bms.withColumn("join_hour", F.date_trunc("hour", col("timestamp"))).groupBy("join_hour").agg(F.collect_list("logs").alias("logs"))


df_realtime_battery_grouped = df_silver_realtime_battery.withColumn("join_hour", F.date_trunc("hour", col("timestamp")))

In [0]:
df_realtime_battery_grouped.display()

In [0]:
df_realtime_bms_grouped.display()

In [0]:
df_silver_realtime_final = df_realtime_battery_grouped.join(df_realtime_bms_grouped, on="join_hour", how="full")
df_silver_realtime_final = df_silver_realtime_final.withColumn(
    "timestamp", 
    F.coalesce(
        F.col("timestamp"), 
        F.col("join_hour")
    )
).drop("join_hour")
df_silver_realtime_final.toPandas().info()

In [0]:
df_silver_realtime_final.display()

In [0]:
df_silver_final = df_silver_old_final.union(df_silver_realtime_final)
df_silver_final.toPandas().info()

### Data Processing

In [0]:
# Window to get last known value
window_prev = Window.orderBy("timestamp")\
                    .rowsBetween(Window.unboundedPreceding, -1)

# Window to get next known value                    
window_next = Window.orderBy("timestamp")\
                    .rowsBetween(1, Window.unboundedFollowing)


df_neighbour = df_silver_final.withColumn("prev_v", F.last("voltage", ignorenulls=True).over(window_prev))\
                              .withColumn("next_v", F.first("voltage", ignorenulls=True).over(window_next))\
                              .withColumn("prev_t", F.last("temperature", ignorenulls=True).over(window_prev))\
                              .withColumn("next_t", F.first("temperature", ignorenulls=True).over(window_next))

df_neighbour.display()

In [0]:
df_final = df_neighbour.withColumn("voltage", F.coalesce(F.col("voltage"), (F.col("prev_v") + F.col("next_v")) / 2, F.col("prev_v"), F.col("next_v")))\
                       .withColumn("temperature", F.coalesce(F.col("temperature"), (F.col("prev_t") + F.col("next_t")) / 2, F.col("prev_t"), F.col("next_t"))).orderBy("timestamp").drop("battery_id", "cycle_count","prev_v", "next_v", "prev_t", "next_t")

df_final.display()

In [0]:
df_final.filter((F.col("temperature") > -20) & (F.col("temperature") < 60)).count()


In [0]:
from pyspark.sql.functions import isnan

df_final_clean = df_final.dropna()
for col_name in df_final_clean.columns:
    df_final_clean = df_final_clean.filter(col(col_name).isNotNull())
df_final_clean.display()

In [0]:
from pyspark.sql.types import FloatType

df_silver = df_final_clean.filter(
    (F.col("temperature") > -20) & 
    (F.col("temperature") < 60) & 
    (F.col("voltage") >= 3) & 
    (F.col("voltage") <= 4.35)
).withColumn(
    "voltage", F.round(F.col("voltage"), 2)
).withColumn(
    "temperature", F.round(F.col("temperature"), 2)
)

df_silver.display()